In [ ]:
import pandas as pd
import numpy as np

# ====================== CONFIG ======================
FILE_PATH = "Copy of Master Data_290102026 2 - Copy.xlsx"
SHEET_NAME = "Master Data "

TARGET_COVERAGE_DAYS = 3.0
MAX_PARTS_PER_MACHINE = 3
MAX_HOURS_PER_DAY = 22.0
CHANGEOVER_HOURS = 40 / 60.0

RUNNER_THRESHOLD = 2000
REPEATER_THRESHOLD = 200

ALLOWED_MACHINES = ["MP-01", "MP-04", "MP-05", "MP-08", "MP-10", "MP-11", "MP-17", "TOYO-IST"]

# ====================== MACHINE CLEANING ======================
def clean_machine(raw):
    if pd.isna(raw) or not str(raw).strip():
        return None
    s = str(raw).strip().upper()
    s = s.replace(".", "-").replace("M.P-", "MP-").replace("MP.", "MP-")
    s = s.replace("TOYOI", "TOYO").replace("TOYO ", "TOYO-").replace("TOYOIST", "TOYO-IST")
    s = s.replace(" ", "-")
    return s

def get_machines(cell):
    if pd.isna(cell):
        return []
    items = str(cell).split(",")
    cleaned = [clean_machine(x) for x in items if clean_machine(x)]
    return list(set(cleaned))

# ====================== LOAD & PROCESS ======================
print("Reading file...")
master = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

# Convert to numeric
for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity", "Cycle Time"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

print("Processing rows (following your exact logic)...")
valid_parts = []

for child, group in master.groupby("Child Part"):
    # 1. Check Daily Plan
    daily_plan_sum = (group["Daily Plan"] * group["Sub Count"]).sum()
    if daily_plan_sum <= 0:
        continue

    # 2. Calculate Net Required Qty (your exact formula)
    min_qty = group["Minimum Quantity"].iloc[0]          # once only
    inventory = group["Inventory_25"].iloc[0]
    net_required = daily_plan_sum + min_qty - inventory
    if net_required <= 0:
        continue

    # 3. Check Vertical Machines
    machines_raw = group["Vertical Machines"].dropna().unique()
    vm_str = ",".join(machines_raw.astype(str))
    eligible = get_machines(vm_str)
    if not eligible:
        continue

    cycle_time = group["Cycle Time"].iloc[0]

    valid_parts.append({
        "Child Part": child,
        "Daily_Demand": daily_plan_sum,
        "Net_Required": net_required,
        "Cycle_Time_sec": cycle_time,
        "Eligible_Machines": eligible,
        "Category": "Unknown"
    })

df = pd.DataFrame(valid_parts)

# Classify
df["Category"] = "Stranger"
df.loc[df["Daily_Demand"] >= RUNNER_THRESHOLD, "Category"] = "Runner"
df.loc[(df["Daily_Demand"] >= REPEATER_THRESHOLD) & 
       (df["Daily_Demand"] < RUNNER_THRESHOLD), "Category"] = "Repeater"

print(f"\nProcessed {len(df):,} valid Child Parts")
print(df["Category"].value_counts())

# Only plan Repeater + Stranger
to_schedule = df[df["Category"].isin(["Repeater", "Stranger"])].copy()

if len(to_schedule) == 0:
    print("No Repeater or Stranger parts found after filtering.")
    exit()

print("\nStarting scheduler...")

machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_parts = {m: [] for m in ALLOWED_MACHINES}
schedule = []

# PHASE 1 + 2 combined (Daily first, then buffer)
for _, part in to_schedule.iterrows():
    hrs_per_pc = part["Cycle_Time_sec"] / 3600.0
    if hrs_per_pc <= 0:
        continue

    qty_to_plan = part["Net_Required"]                     # full net (daily + buffer)
    eligible = [m for m in part["Eligible_Machines"] if m in ALLOWED_MACHINES]
    if not eligible:
        continue

    # Try to assign as big lot (3-day style)
    for m in sorted(eligible, key=lambda x: machine_load[x]):
        if len(machine_parts[m]) >= MAX_PARTS_PER_MACHINE:
            continue

        free = MAX_HOURS_PER_DAY - machine_load[m]
        if free <= CHANGEOVER_HOURS:
            continue

        setup_h = CHANGEOVER_HOURS
        free_after = free - setup_h
        if free_after <= 0:
            continue

        max_qty = free_after / hrs_per_pc
        assign_qty = min(qty_to_plan, max_qty)
        if assign_qty < 10:
            continue

        assign_h = assign_qty * hrs_per_pc

        machine_load[m] += assign_h + setup_h
        machine_parts[m].append({
            "Child Part": part["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Type": "Planned"
        })

        schedule.append({
            "Machine": m,
            "Child Part": part["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Type": "Planned"
        })

        qty_to_plan -= assign_qty
        if qty_to_plan <= 0:
            break

# ====================== OUTPUT ======================
print("\n" + "="*90)
print("FINAL 3-DAY BUFFER PLAN (following your exact logic)")
print("="*90)

total_hours = sum(machine_load.values())
total_changeovers = 0

for m in sorted(ALLOWED_MACHINES):
    parts = machine_parts.get(m, [])
    hours = machine_load.get(m, 0.0)
    if hours == 0 and not parts:
        continue

    chg = len(parts) - 1 if len(parts) > 1 else 0   # conservative count
    total_changeovers += chg

    print(f"\n🛠 {m}   {hours:6.1f} / 22.0 h   ({hours/22*100:5.1f}%)   {len(parts)} parts")
    for p in parts:
        print(f"   • {p['Child Part']:20}   {p['Qty']:>6} pcs   {p['Hours']:>5.1f}h   {p['Type']}")

print("\n" + "-"*90)
print(f"Total hours used     : {total_hours:.1f} h")
print(f"Estimated changeovers: {total_changeovers}")
print(f"Parts scheduled      : {len(schedule)}")
print("-"*90)

pd.DataFrame(schedule).to_excel("final_3day_plan.xlsx", index=False)
print("Saved → final_3day_plan.xlsx")

In [ ]:
import pandas as pd
import numpy as np
import re

# ===============================================
# CONFIGURATION
# ===============================================
FILE_PATH = "Copy of Master Data_290102026 2 - Copy.xlsx"
SHEET_NAME = "Master Data " # Ensure this exact name matches your sheet

ALLOWED_120T = ["MP-01", "MP-05", "MP-10", "MP-17"]
MAX_HOURS = 22.0
CHANGEOVER_HOURS = 40 / 60  # 40 minutes

def generate_production_plan():
    # 1. LOAD DATA
    try:
        df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    except Exception as e:
        return f"Error loading file: {e}"

    # 2. CLEAN & FILTER
    # Only keep rows where Daily Plan is not 0 or NaN
    df['Daily Plan'] = pd.to_numeric(df['Daily Plan'], errors='coerce').fillna(0)
    df = df[df['Daily Plan'] > 0].copy()

    # 3. AGGREGATE CHILD PARTS
    # Calculation: Σ (Daily Plan * Sub Count)
    df['Requirement_Contribution'] = df['Daily Plan'] * df['Sub Count']
    
    # Group by Child Part to ensure Minimum Quantity is added only ONCE
    agg_logic = {
        'Requirement_Contribution': 'sum',
        'Minimum Quantity': 'first',
        'Inventory_25': 'first',
        'Cycle Time': 'first',
        'Vertical Machines': 'first',
        'Category': 'first'
    }
    
    child_agg = df.groupby('Child Part').agg(agg_logic).reset_index()

    # 4. CALCULATE NET REQUIRED QTY
    # Net_Required_Qty = Σ(Demand) + Min_Qty - Inventory
    child_agg['Net_Required_Qty'] = (
        child_agg['Requirement_Contribution'] + 
        child_agg['Minimum Quantity'] - 
        child_agg['Inventory_25']
    ).clip(lower=0)

    # 5. FILTER FOR 120T MACHINES
    def is_120t(m_str):
        if pd.isna(m_str): return []
        tokens = re.split(r"[,|/\\\s]+", str(m_str).upper())
        # Clean tokens to match "MP-XX" format
        cleaned = [t.replace("MP", "MP-") if "MP" in t and "-" not in t else t for t in tokens]
        return [m for m in cleaned if m in ALLOWED_120T]

    child_agg['Eligible_120T'] = child_agg['Vertical Machines'].apply(is_120t)
    plan_data = child_agg[child_agg['Eligible_120T'].map(len) > 0].copy()

    # 6. PRIORITIZE BY CATEGORY
    prio_map = {'Runner': 1, 'Repeater': 2, 'Stranger': 3}
    plan_data['Prio'] = plan_data['Category'].map(prio_map).fillna(4)
    plan_data = plan_data.sort_values(by=['Prio', 'Net_Required_Qty'], ascending=[True, False])

    # 7. ALLOCATION
    machine_loads = {m: 0.0 for m in ALLOWED_120T}
    final_plan = []

    for _, row in plan_data.iterrows():
        qty_needed = row['Net_Required_Qty']
        ct_sec = row['Cycle Time']
        if qty_needed <= 0 or ct_sec <= 0: continue
        
        hrs_needed = (qty_needed * ct_sec) / 3600
        
        for m in row['Eligible_120T']:
            if hrs_needed <= 0: break
            
            space = MAX_HOURS - machine_loads[m]
            if space <= CHANGEOVER_HOURS: continue
            
            allocated_hrs = min(space, hrs_needed)
            allocated_qty = (allocated_hrs * 3600) / ct_sec
            
            machine_loads[m] += allocated_hrs
            hrs_needed -= allocated_hrs
            
            final_plan.append({
                "Machine": m,
                "Child Part": row["Child Part"],
                "Category": row["Category"],
                "Quantity": int(allocated_qty),
                "Time Taken (Hrs)": round(allocated_hrs, 2)
            })

    # 8. RESULTS
    output_df = pd.DataFrame(final_plan)
    
    print("\n--- 120T MACHINE LOAD SUMMARY ---")
    for m, load in machine_loads.items():
        print(f"{m}: {load:.2f} / {MAX_HOURS} hrs ({(load/MAX_HOURS)*100:.1f}%)")
        
    return output_df

if __name__ == "__main__":
    plan = generate_production_plan()
    if isinstance(plan, pd.DataFrame) and not plan.empty:
        plan.to_excel("120T_Production_Plan.xlsx", index=False)
        print("\nSuccess: Plan saved to 120T_Production_Plan.xlsx")
    else:
        print("\nNo parts were eligible for the current 120T plan.")